<a href="https://colab.research.google.com/github/mohsen-alshoklia/Manim-CE-files/blob/main/SumVsIntegration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt update
!sudo apt install libcairo2-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science tipa libpango1.0-dev
!pip install manim

In [17]:
from manim import *
import numpy as np
from IPython.display import Video, display
import subprocess

class SumVsIntegration(Scene):
    def construct(self):
        # --- 1. Top Text Elements ---
        title = Text("Sum vs Integration", font_size=50, weight=BOLD)
        title.set_color_by_gradient(PINK, YELLOW)
        title.move_to(6 * UP)

        equation = MathTex(r"y = \sin^2(x) \cos(x)", color=BLUE_C, font_size=36)
        equation.next_to(title, DOWN, buff=0.5)

        # --- 2. Setup Axes and Graph ---
        axes = Axes(
            x_range=[0, 2, 0.5],     # Go up to a bit past PI/2 (approx 1.57)
            y_range=[0, 0.45, 0.1],  # Max height is around 0.38
            x_length=7,
            y_length=5,
            axis_config={"color": WHITE},
            tips=True
        )
        axes.move_to(1.5 * UP)

        # Axis labels
        x_label = axes.get_x_axis_label("x")
        y_label = axes.get_y_axis_label("y")

        # The mathematical function
        def func(x):
            return (np.sin(x)**2) * np.cos(x)

        graph = axes.plot(func, x_range=[0, PI/2], color=BLUE_C, stroke_width=3)

        # --- 3. Dynamic Values with Midpoint Rule and Gradient Colors ---
        n_tracker = ValueTracker(4) # Start with 4 rectangles

        # Custom function to create rectangles with gradient colors using midpoint rule
        def get_midpoint_rectangles_with_gradient():
            n = int(n_tracker.get_value())
            dx = (PI/2) / n

            rectangles = VGroup()

            # Define gradient colors
            start_color = GREEN
            end_color = BLUE

            for i in range(n):
                # Calculate midpoint for this rectangle
                x_mid = (i + 0.5) * dx

                # Calculate rectangle position and dimensions
                y_val = func(x_mid)

                # Create rectangle
                rect = Rectangle(
                    width=axes.x_length * dx / (axes.x_range[1] - axes.x_range[0]),
                    height=axes.y_length * y_val / (axes.y_range[1] - axes.y_range[0]),
                    stroke_width=0.5,
                    stroke_color=WHITE
                )

                # Position the rectangle (centered at the midpoint)
                rect.move_to(axes.c2p(x_mid, y_val/2))

                # Calculate color based on position (gradient from left to right)
                alpha = i / max(n - 1, 1)  # Progress from 0 to 1
                color = interpolate_color(start_color, end_color, alpha)

                # Apply fill color and opacity
                rect.set_fill(color, opacity=0.35)

                rectangles.add(rect)

            return rectangles

        # Updaters for Rectangles using midpoint rule with gradients
        rects = always_redraw(get_midpoint_rectangles_with_gradient)

        # Helper function to calculate the numeric sum using midpoint rule
        def get_current_sum():
            n = int(n_tracker.get_value())
            dx = (PI/2) / n
            # Use midpoints for sum calculation
            x_vals = np.linspace(dx/2, PI/2 - dx/2, n)
            return sum(func(x) * dx for x in x_vals)

        # --- 4. Bottom UI Panels ---

        # Panel 1: The Sum (Updated to indicate midpoint rule)
        sum_math = MathTex(r"\sum \Delta x f(x_{\text{mid}}) \approx ", font_size=36)
        sum_val = always_redraw(lambda: DecimalNumber(get_current_sum(), num_decimal_places=7, font_size=36).next_to(sum_math, RIGHT))
        sum_group = VGroup(sum_math, sum_val).move_to(2.5 * DOWN)
        sum_box = always_redraw(lambda: SurroundingRectangle(sum_group, color=YELLOW, buff=0.2))

        # Panel 2: The 'n' counter
        n_math = MathTex("n = ", font_size=36)
        n_val = always_redraw(lambda: Integer(int(n_tracker.get_value()), font_size=36).next_to(n_math, RIGHT))
        n_group = VGroup(n_math, n_val).next_to(sum_group, DOWN, buff=0.8)
        n_box = always_redraw(lambda: SurroundingRectangle(n_group, color=TEAL, buff=0.2))

        # Panel 3: The Exact Integral
        integral_math = MathTex(r"\int_0^{\pi/2} \sin^2(x) \cos(x) dx = \frac{1}{3}", font_size=36)
        integral_math.next_to(n_group, DOWN, buff=0.8)
        integral_box = SurroundingRectangle(integral_math, color=PURPLE, buff=0.2)

        # Add a label for the method
        method_label = Text("Midpoint Rule", font_size=24, color=YELLOW)
        method_label.next_to(axes, UP, buff=0.1)
        method_label.shift(RIGHT * 2)

        # Add a checkmark to show convergence
        check = Text("✓", font_size=40, color=GREEN)
        check.next_to(sum_box, RIGHT, buff=0.3)

        # --- 5. Animation Sequence ---

        # Intro (approx 2 seconds)
        self.add(title, method_label)
        self.play(Create(axes), FadeIn(x_label, y_label))
        self.play(Create(graph), Write(equation))
        self.wait(0.5)

        # Bring in the initial state (n=4)
        self.play(
            FadeIn(rects),
            Write(sum_group), Create(sum_box),
            Write(n_group), Create(n_box),
            Write(integral_math), Create(integral_box),
            run_time=1
        )
        self.wait(0.5)

        # The big transition: ramp up 'n' from 4 to 99 over 8.5 seconds
        self.play(
            n_tracker.animate.set_value(99),
            run_time=8.5,
            rate_func=linear # Linear keeps the counting speed consistent
        )

        self.wait(0.5)

        # --- 7. Show Checkmark Animation ---
        self.play(Write(check), run_time=0.5)

        self.wait(0.5)

# --- Force Vertical Rendering (The DeepSeek Method) ---
if __name__ == "__main__":
    from manim import config as manim_config

    # Override config for 9:16 vertical
    manim_config.frame_width = 9
    manim_config.frame_height = 16
    manim_config.pixel_height = 1920
    manim_config.pixel_width = 1080

    scene = SumVsIntegration()
    scene.render()

    video_path = scene.renderer.file_writer.movie_file_path

    # Display the video in Colab
    display(Video(video_path, embed=True))